In [9]:
# 関数の定義
# CAM部
def calc_cam_oprations(
        neuron_num: int,
        input_dim: int,
        cam_bit: int,
        odeu: int
):
    entries = 2 ** cam_bit
    # ビット数
    cam_bits = entries * cam_bit
    # 演算回数
    cam_oprations = odeu * (neuron_num + input_dim) * entries

    return cam_bits, cam_oprations

# LUT部
def calc_lut_oprations(
        neuron_num: int,
        input_dim: int,
        cam_bit: int,
        lut_bit: int,
        odeu: int
):
    entries = 2 ** cam_bit
    # ビット数
    lut_bits = entries * neuron_num * lut_bit

    # 演算回数
    lut_oprations = odeu * (neuron_num + input_dim) * neuron_num

    return lut_bits, lut_oprations

# MVM部
def calc_mvm_oprations(
        neuron_num: int,
        input_dim: int,
        mvm_bit: int,
        odeu: int
):
    # ビット数
    mvm_bits = neuron_num * (neuron_num + input_dim) * mvm_bit

    # 演算回数 (Add/Mul)
    mvm_oprations = 2 * odeu * (neuron_num + input_dim) * neuron_num

    return mvm_bits, mvm_oprations

# Arithmetic部
def calc_arithmetic_oprations(
        neuron_num: int,
        arithmetic_bit_RRAM: int,
        arithmetic_bit_SRAM: int,
        odeu: int
):
    # ビット数 (保存用)
    arithmetic_bits = 3 * neuron_num * arithmetic_bit_RRAM + neuron_num * arithmetic_bit_SRAM

    # 演算回数 (Add/Mul/Div)
    arithmetic_oprations = 6 * odeu * neuron_num

    return arithmetic_bits, arithmetic_oprations


In [11]:
# 全体のビット数と演算量の計算と可視化
def calc_total_oprations(
        neuron_num: int,
        input_dim: int,
        cam_bit: int,
        lut_bit: int,
        mvm_bit: int,
        arithmetic_bit_RRAM: int,
        arithmetic_bit_SRAM: int,
        odeu: int
):
    cam_bits, cam_oprations = calc_cam_oprations(
        neuron_num,
        input_dim,
        cam_bit,
        odeu
    )

    lut_bits, lut_oprations = calc_lut_oprations(
        neuron_num,
        input_dim,
        cam_bit,
        lut_bit,
        odeu
    )

    mvm_bits, mvm_oprations = calc_mvm_oprations(
        neuron_num,
        input_dim,
        mvm_bit,
        odeu
    )

    arithmetic_bits, arithmetic_oprations = calc_arithmetic_oprations(
        neuron_num,
        arithmetic_bit_RRAM,
        arithmetic_bit_SRAM,
        odeu
    )

    total_bits = cam_bits + lut_bits + mvm_bits + arithmetic_bits
    total_oprations = cam_oprations + lut_oprations + mvm_oprations + arithmetic_oprations

    return {
        "CAM": {"bits": cam_bits, "operations": cam_oprations},
        "LUT": {"bits": lut_bits, "operations": lut_oprations},
        "MVM": {"bits": mvm_bits, "operations": mvm_oprations},
        "Arithmetic": {"bits": arithmetic_bits, "operations": arithmetic_oprations},
        "Total": {"bits": total_bits, "operations": total_oprations}
    }

import pandas as pd

res = calc_total_oprations(
    neuron_num=256,
    input_dim=19,
    cam_bit=8,
    lut_bit=8,
    mvm_bit=8,
    arithmetic_bit_RRAM=8,
    arithmetic_bit_SRAM=8,
    odeu=3
)

rows = []
for k in ["CAM", "LUT", "MVM", "Arithmetic"]:
    rows.append({
        "Module": k,
        "Bits": res[k]["bits"],
        "Bit Ratio [%]": 100 * res[k]["bits"] / res["Total"]["bits"],
        "Operations": res[k]["operations"],
        "Op Ratio [%]": 100 * res[k]["operations"] / res["Total"]["operations"],
    })

df = pd.DataFrame(rows)

# Total 行を追加
df.loc[len(df)] = {
    "Module": "Total",
    "Bits": res["Total"]["bits"],
    "Bit Ratio [%]": 100.0,
    "Operations": res["Total"]["operations"],
    "Op Ratio [%]": 100.0,
}

df


,Module,Bits,Bit Ratio [%],Operations,Op Ratio [%]
0,CAM,2048,0.186567,211200,24.864376
1,LUT,524288,47.761194,211200,24.864376
2,MVM,563200,51.305970,422400,49.728752
3,Arithmetic,8192,0.746269,4608,0.542495
4,Total,1097728,100.000000,849408,100.000000


In [8]:
# GPU
def calc_GPU(units, input_dim, discrete_step):
    Sigmoid_bits = units * (units + input_dim) * 2 * 32  # mu, sigma
    sigmoid_ops_add = units * (units + input_dim) * 2 * discrete_step
    sigmoid_ops_mul = units * (units + input_dim) * (discrete_step + 1)
    sigmoid_ops_ex2 = units * (units + input_dim) 
    sigmoid_ops_div = units * (units + input_dim) * discrete_step
    print("Sigmoid_bits:" , Sigmoid_bits, "energy:" , Sigmoid_bits * 3.9e-12)
    print("sigmoid_ops_add:" , sigmoid_ops_add, "energy:" , sigmoid_ops_add * 1.2e-9)
    print("sigmoid_ops_mul:" , sigmoid_ops_mul, "energy:" , sigmoid_ops_mul * 1.4e-9)
    print("sigmoid_ops_ex2:" , sigmoid_ops_ex2, "energy:" , sigmoid_ops_ex2 * 4.71e-7)
    print("sigmoid_ops_div:" , sigmoid_ops_div, "energy:" , sigmoid_ops_div * 5.04e-6)
    print("Total energy:" , (Sigmoid_bits * 3.9e-12) + (sigmoid_ops_add * 1.2e-9) + (sigmoid_ops_mul * 1.4e-9) + (sigmoid_ops_ex2 * 4.71e-7) + (sigmoid_ops_div * 5.04e-6))

    MVM_bits = units * (units + input_dim) * 3 * 32 # w, mask, erev
    MVM_ops_ready_mul = 2 * (units + input_dim) * units # preprare for w, w_ij
    MVM_ops_mul = (units + input_dim) * (units + input_dim)
    MVM_ops_add = (units + input_dim) * (units + input_dim)
    print("MVM_bits:" , MVM_bits, "energy:" , MVM_bits * 3.9e-12)
    print("MVM_ops_mul:" , MVM_ops_ready_mul + MVM_ops_mul, "energy:" , (MVM_ops_ready_mul + MVM_ops_mul) * 1.4e-9)
    print("MVM_ops_add:" , MVM_ops_add, "energy:" , MVM_ops_add * 1.2e-9)
    print("Total energy:" , (MVM_bits * 3.9e-12) + ((MVM_ops_ready_mul + MVM_ops_mul) * 1.4e-9) + (MVM_ops_add * 1.2e-9))

    Arith_bits = (units + input_dim) * 3 * 32 # 3 #gleak,vleak,cm
    Arith_ops_add = units * 4
    Arith_ops_mul = units 
    Arith_ops_div = units
    print("Arith_bits:" , Arith_bits, "energy:" , Arith_bits * 3.9e-12)
    print("Arith_ops_add:" , Arith_ops_add, "energy:" , Arith_ops_add * 1.2e-9)
    print("Arith_ops_mul:" , Arith_ops_mul, "energy:" , Arith_ops_mul * 1.4e-9)
    print("Arith_ops_div:" , Arith_ops_div, "energy:" , Arith_ops_div * 5.04e-6)
    print("Total energy:" , (Arith_bits * 3.9e-12) + (Arith_ops_add * 1.2e-9) + (Arith_ops_mul * 1.4e-9) + (Arith_ops_div * 5.04e-6))
calc_GPU(256, 19, 3)

Sigmoid_bits: 4505600 energy: 1.757184e-05
sigmoid_ops_add: 422400 energy: 0.00050688
sigmoid_ops_mul: 281600 energy: 0.00039423999999999997
sigmoid_ops_ex2: 70400 energy: 0.033158400000000005
sigmoid_ops_div: 211200 energy: 1.064448
Total energy: 1.09852509184
MVM_bits: 6758400 energy: 2.635776e-05
MVM_ops_mul: 216425 energy: 0.00030299499999999997
MVM_ops_add: 75625 energy: 9.075e-05
Total energy: 0.00042010276
Arith_bits: 26400 energy: 1.0296e-07
Arith_ops_add: 1024 energy: 1.2288e-06
Arith_ops_mul: 256 energy: 3.584e-07
Arith_ops_div: 256 energy: 0.00129024
Total energy: 0.00129193016
